# R5.2 — Ensemble-based baselines on our noisy labels

Two-network methods that expose a clean/noisy partition, run on the exact published noisy
labels and scored with our detection/correction metrics: **Co-teaching** (Han et al. 2018,
reimplemented; the official code targets PyTorch 0.3) and **DivideMix** (Li et al. 2020, the
official code at a pinned commit, minimally patched -- every change is listed in
`r52_patch.diff`, copied into each job folder). Jobs pause with a checkpoint before the
12-hour limit and resume in the next version. Classic ensemble filters (majority, consensus,
Confident Learning) are computed from the R5.1 member outputs offline (`r52_filters.py`).
See `kaggle/README.md` for detection/correction rules and runtimes.

**Kaggle settings:** Accelerator *GPU T4 x2* (P100 also works, one worker), Internet *On*
(phone-verified account), Persistence *Files only*. Run with **Save Version -> Save & Run All
(Commit)** so it runs in the background for up to 12 h.

**Resuming:** jobs are saved as they finish. If the session ends with jobs left, open the
notebook, *Add Input -> Your Work -> this notebook* (the previous version's output), and
commit again: finished jobs are restored and skipped. Repeat until the launch cell prints
that nothing is left; the analysis cell then reports the final tables.

**Outputs** (in the version's Output tab): `r52_out/` raw per-job results, `results/` CSV
tables, `logs/` per-GPU training logs.

## 1. Configuration

In [ ]:
import json, os, time
SESSION_START = time.time()
CFG = dict(
    jobs=[['dividemix', 'cifar10', 20], ['coteaching', 'cifar10', 20],
          ['coteaching', 'fashionmnist', 20], ['dividemix', 'fashionmnist', 20]],
    worker_prefs={'0': ['dividemix', 'coteaching'], '1': ['coteaching', 'dividemix']},
    preds_ref='main', code_ref='main', outer_folds=[1], seed=0, amp=True,
    session_hours=11.5, min_job_hours=0.5,
    coteaching={}, dividemix={'num_workers': 3},
    # our row in the table: the oracle thresholds of the run named by preds_ref
    ours_thresholds={'cifar10_20': [10, 10], 'cifar10_30': [8, 10], 'cifar10_40': [8, 10]},
    r51_dirs={},
)
CFG.update(json.loads(os.environ.get('SND_CONFIG_OVERRIDES', '{}')))   # local smoke tests only

## 2. Setup (clone the repo, import kaggle/src, prepare data)

In [ ]:
import subprocess, sys
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd())
subprocess.run('nvidia-smi -L', shell=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'cleanlab'], check=False)
REPO = os.environ.get('SND_REPO') or '/tmp/snd/SiameseNoiseDetection'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/stormaref/SiameseNoiseDetection.git',
                    REPO], check=True)
if not os.environ.get('SND_REPO'):
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', CFG['code_ref']], check=True)
SRC = os.path.join(REPO, 'kaggle', 'src')
sys.path.insert(0, SRC)
import snd_kaggle as K
COMMIT = subprocess.run(['git', '-C', REPO, 'log', '-1', '--format=%H %ad %s', '--date=iso'],
                        capture_output=True, text=True).stdout.strip()
print('code:', COMMIT)
CFG['out_dir'] = K.restore_previous_outputs('r52_out')
CFG.update(session_start=SESSION_START, commit=COMMIT)
CONFIG = os.path.abspath('r52_config.json')
json.dump(CFG, open(CONFIG, 'w'), indent=1)
json.dump(CFG, open(os.path.join(CFG['out_dir'], f'config_{int(SESSION_START)}.json'), 'w'), indent=1)
subprocess.run([sys.executable, os.path.join(SRC, 'r52_worker.py'), CONFIG, '--prepare'], check=True)

## 3. Train (all GPUs, resumable)

In [ ]:
codes = K.launch_workers(os.path.join(SRC, 'r52_worker.py'), CONFIG, poll=CFG.get('poll', 300))
print('worker exit codes:', codes)
for s in K.JobQueue(CFG['out_dir']).summaries():
    print(s)

## 4. Analysis

In [ ]:
subprocess.run([sys.executable, os.path.join(SRC, 'r52_analysis.py'), CONFIG], check=False)